# Daily Solar Yield Inference

Loads a trained MLflow model, scores the latest Open-Meteo forecast, applies physical guardrails, and projects the result into plane-of-array GTI.


In [0]:
%pip install openmeteo-requests pvlib pandas pyarrow xgboost scikit-learn seaborn requests
dbutils.library.restartPython()

In [ ]:
from pathlib import Path
import sys

import mlflow.sklearn
import openmeteo_requests
import pandas as pd


def add_repo_src_to_path():
    candidates = []
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
        workspace_dir = workspace_path.parent
        candidates.extend([workspace_dir, *workspace_dir.parents])
    except Exception:
        pass

    for candidate in candidates:
        src_path = candidate / "src"
        if (src_path / "solar_yield" / "features.py").exists():
            src_path_text = str(src_path)
            if src_path_text not in sys.path:
                sys.path.insert(0, src_path_text)
            return candidate

    raise RuntimeError("Could not locate src/solar_yield/features.py from this notebook.")


repo_root = add_repo_src_to_path()
print(f"Using shared package source from: {repo_root / 'src'}")

from solar_yield.features import (  # noqa: E402
    FEATURE_COLUMNS,
    add_model_features,
    apply_physical_overrides,
    prepare_model_matrix,
)
from solar_yield.quality import validate_hourly_forecast  # noqa: E402

# ==========================================
# 1. ARCHITECTURE SETUP & CONFIGURATION
# ==========================================
dbutils.widgets.text("model_uri", "", "Exact MLflow model URI")
dbutils.widgets.text("model_run_id", "", "MLflow model run ID fallback")
MODEL_URI = dbutils.widgets.get("model_uri").strip()
RUN_ID = dbutils.widgets.get("model_run_id").strip()
if not MODEL_URI:
    if not RUN_ID:
        raise ValueError(
            "Set model_uri from the training notebook output, or set model_run_id "
            "for fallback runs:/ loading."
        )
    MODEL_URI = f"runs:/{RUN_ID}/solar_factor_model"

print(f"Loading production time-aware weighted chained model from MLflow: {MODEL_URI}")
model = mlflow.sklearn.load_model(MODEL_URI)

# ==========================================
# 2. BRONZE LAYER: LIVE FORECAST INGESTION
# ==========================================
om = openmeteo_requests.Client()
params = {
    "latitude": -31.95,
    "longitude": 115.86,
    "hourly": [
        "cloud_cover_low",
        "cloud_cover_mid",
        "cloud_cover_high",
        "total_column_integrated_water_vapour",
        "sunshine_duration",
        "temperature_2m",
        "relative_humidity_2m",
        "surface_pressure",
    ],
    "timezone": "Australia/Perth",
    "forecast_days": 7,
}
responses = om.weather_api("https://api.open-meteo.com/v1/forecast", params=params)
hourly = responses[0].Hourly()

start_epoch = hourly.Time()
end_epoch = hourly.TimeEnd()
step_seconds = hourly.Interval()

date_range = pd.date_range(
    start=pd.to_datetime(start_epoch, unit="s"),
    end=pd.to_datetime(end_epoch, unit="s"),
    freq=pd.Timedelta(seconds=step_seconds),
    inclusive="left",
)

print(f"Generated timestamp vector length: {len(date_range)}")
print(f"Generated weather feature length: {len(hourly.Variables(0).ValuesAsNumpy())}")

pdf_forecast_raw = pd.DataFrame(
    {
        "timestamp": date_range,
        "cloud_low": hourly.Variables(0).ValuesAsNumpy(),
        "cloud_mid": hourly.Variables(1).ValuesAsNumpy(),
        "cloud_high": hourly.Variables(2).ValuesAsNumpy(),
        "water_vapour": hourly.Variables(3).ValuesAsNumpy(),
        "sunshine_duration": hourly.Variables(4).ValuesAsNumpy(),
        "temperature": hourly.Variables(5).ValuesAsNumpy(),
        "relative_humidity": hourly.Variables(6).ValuesAsNumpy(),
        "surface_pressure": hourly.Variables(7).ValuesAsNumpy(),
    }
)

pdf_forecast_raw["timestamp"] = pdf_forecast_raw["timestamp"].dt.tz_localize(None)
validate_hourly_forecast(pdf_forecast_raw, expected_rows=168)

# Keep a Bronze Spark frame for Databricks layer visibility and optional inspection.
df_forecast_bronze = spark.createDataFrame(pdf_forecast_raw)

# ==========================================
# 3. SILVER LAYER: SHARED TIME-AWARE FEATURES
# ==========================================
print("Engineering shared time-aware inference features...")
X_inference_full = add_model_features(pdf_forecast_raw, mode="inference")
X_features = prepare_model_matrix(X_inference_full, FEATURE_COLUMNS)

if len(X_features) != 168:
    raise ValueError(f"expected 168 inference feature rows, got {len(X_features)}")
if X_features.isna().any().any():
    raise ValueError("inference feature matrix contains null values")

print(f"Inference feature matrix shape: {X_features.shape}")


def first_estimator_feature_names(loaded_model):
    estimators = getattr(loaded_model, "estimators_", [])
    if not estimators:
        return None
    estimator = estimators[0]
    feature_names = getattr(estimator, "feature_names_in_", None)
    if feature_names is not None:
        return list(feature_names)
    booster_getter = getattr(estimator, "get_booster", None)
    if booster_getter is not None:
        booster = booster_getter()
        if getattr(booster, "feature_names", None):
            return list(booster.feature_names)
    return None


expected_feature_names = first_estimator_feature_names(model)
if expected_feature_names is not None:
    missing = [column for column in expected_feature_names if column not in X_features.columns]
    unexpected = [column for column in X_features.columns if column not in expected_feature_names]
    print(f"Loaded model first-estimator feature count: {len(expected_feature_names)}")
    if missing or unexpected:
        raise ValueError(
            "Loaded model feature schema does not match the inference feature matrix. "
            "Use the exact model_uri printed by the Phase 3 training notebook. "
            f"Missing columns: {missing[:10]}; unexpected columns: {unexpected[:10]}"
        )

# Keep a Silver Spark frame for Databricks layer visibility and optional inspection.
df_forecast_silver = spark.createDataFrame(X_inference_full)

# ==========================================
# 4. GOLD LAYER: INFERENCE & PHYSICAL OVERRIDES
# ==========================================
print("Generating predictions and applying physical overrides...")
raw_predictions = model.predict(X_features)
X_inference_full = apply_physical_overrides(X_inference_full, raw_predictions)

# Convert finalized arrays to Spark Gold Layer for production storage/BI dashboarding.
df_forecast_gold = spark.createDataFrame(
    X_inference_full[
        [
            "timestamp",
            "temperature",
            "sunshine_fraction",
            "final_pred_direct",
            "final_pred_diffuse",
        ]
    ]
)

# Write out to your production Delta lake directory.
# df_forecast_gold.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.default.solar_power_7d_forecast")

print("==================================================================")
print("SUCCESS: 7-Day Production Forecast Successfully Written to Gold!")
print(f"Generated {df_forecast_gold.count()} rows of clear sky solar attenuation vectors.")
print("==================================================================")


In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Bring the dataset locally, sort, and strip timezone structures
pdf_fixed = df_forecast_gold.orderBy("timestamp").toPandas()
pdf_fixed["timestamp"] = pd.to_datetime(pdf_fixed["timestamp"].dt.tz_localize(None))

# Create a clean string column for labels (e.g., "May 26 12:00")
pdf_fixed["time_str"] = pdf_fixed["timestamp"].dt.strftime("%b %d %H:%M")

# 2. Reset the layout - Using an integer range for the X axis to stop the tick error
x_indices = np.arange(len(pdf_fixed))

sns.set_theme(style="whitegrid")
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ------------------------------------------------------------------
# TOP PLOT: Direct Solar Factor
# ------------------------------------------------------------------
axs[0].plot(x_indices, pdf_fixed["final_pred_direct"], color="#1f77b4", lw=2.5, label="Direct Attenuation Factor")
axs[0].fill_between(x_indices, pdf_fixed["final_pred_direct"], color="#1f77b4", alpha=0.1)
axs[0].set_ylabel("Direct Factor", fontsize=11)
axs[0].set_ylim(-0.05, 1.05)
axs[0].set_title("Operational Solar Yield Forecast Profile (Perth Local Time)", fontsize=13, fontweight="bold")
axs[0].legend(loc="upper right")

# ------------------------------------------------------------------
# MIDDLE PLOT: Diffuse Solar Factor
# ------------------------------------------------------------------
axs[1].plot(x_indices, pdf_fixed["final_pred_diffuse"], color="#ff7f0e", lw=2.5, label="Diffuse Attenuation Factor")
axs[1].fill_between(x_indices, pdf_fixed["final_pred_diffuse"], color="#ff7f0e", alpha=0.1)
axs[1].set_ylabel("Diffuse Factor", fontsize=11)
axs[1].set_ylim(-0.05, 1.05)
axs[1].legend(loc="upper right")

# ------------------------------------------------------------------
# BOTTOM PLOT: Sunshine Fraction
# ------------------------------------------------------------------
axs[2].plot(x_indices, pdf_fixed["sunshine_fraction"], color="#2ca02c", lw=1.5, linestyle="--", label="Sunshine Fraction")
axs[2].set_ylabel("Sunshine Fraction", fontsize=11)
axs[2].set_ylim(-0.05, 1.05)
axs[2].legend(loc="upper right")

# ------------------------------------------------------------------
# MANUAL X-AXIS OVERRIDE: Exactly 7 clean labels over the 7 days
# ------------------------------------------------------------------
# Pick 7 evenly spaced indices across your 336 rows (roughly every 2 days)
tick_indices = np.linspace(0, len(pdf_fixed) - 1, 7, dtype=int)
tick_labels = pdf_fixed["timestamp"].dt.strftime("%b %d").iloc[tick_indices].values

axs[2].set_xticks(tick_indices)
axs[2].set_xticklabels(tick_labels, fontsize=10)

plt.xlabel("7-Day Forecast Window", fontsize=11, labelpad=10)
plt.tight_layout()
plt.show()

In [0]:
import pvlib
from pvlib.location import Location
import pandas as pd
import numpy as np

# 1. Clear Site Geometry
LATITUDE = -31.95
LONGITUDE = 115.86
TZ = "Australia/Perth"
SURFACE_TILT = 25.0     # Ideal tilt for Perth
SURFACE_AZIMUTH = 0.0   # 0 = True North (Southern Hemisphere)
ALBEDO = 0.2           

print("Initializing localized geometry engine...")
site = Location(latitude=LATITUDE, longitude=LONGITUDE, tz=TZ)

# 2. Pull data from Spark and enforce strict Localization
df_gold_predictions = df_forecast_gold.orderBy("timestamp").toPandas()

# CRITICAL FIX: Ensure the index is explicitly localized to Perth AWST for pvlib math
df_gold_predictions["timestamp"] = pd.to_datetime(df_gold_predictions["timestamp"])
if df_gold_predictions["timestamp"].dt.tz is None:
    df_gold_predictions.index = df_gold_predictions["timestamp"].dt.tz_localize(TZ)
else:
    df_gold_predictions.index = df_gold_predictions["timestamp"].dt.tz_convert(TZ)

# 3. Compute precise solar track vectors matching local hours
solar_position = site.get_solarposition(df_gold_predictions.index)
zenith = solar_position['zenith']
apparent_elevation = solar_position['apparent_elevation']
azimuth = solar_position['azimuth']

# 4. Generate the localized clear sky envelopes
clear_sky = site.get_clearsky(df_gold_predictions.index)

# 5. Apply your model's prediction fractions
attenuated_dni = clear_sky['dni'] * df_gold_predictions['final_pred_direct']
attenuated_dhi = clear_sky['dhi'] * df_gold_predictions['final_pred_diffuse']
attenuated_ghi = (attenuated_dni * np.cos(np.radians(zenith))) + attenuated_dhi

# 6. Total Plane-of-Array (POA) GTI calculation
print("Projecting paths onto North-facing planes...")
total_gti = pvlib.irradiance.get_total_irradiance(
    surface_tilt=SURFACE_TILT,
    surface_azimuth=SURFACE_AZIMUTH,
    solar_zenith=zenith,
    solar_azimuth=azimuth,
    dni=attenuated_dni,
    ghi=attenuated_ghi,
    dhi=attenuated_dhi,
    albedo=ALBEDO,
    model='isotropic'
)

# 7. Map variables back to your clean dataframe
df_gold_predictions["gti_total"] = total_gti['poa_global'].values
df_gold_predictions["gti_direct"] = total_gti['poa_direct'].values
df_gold_predictions["gti_diffuse"] = total_gti['poa_diffuse'].values

# Overwrite Gold Spark table with mathematically corrected profiles
df_final_power_forecast = spark.createDataFrame(df_gold_predictions[[
    "timestamp", "temperature", "final_pred_direct", "final_pred_diffuse", 
    "gti_total", "gti_direct", "gti_diffuse"
]])

print("=" * 65)
print("SUCCESS: LOCALIZED GEOMETRIC CORRECTION COMPLETE!")
print(f"Peak predicted 7-day GTI array yield: {df_gold_predictions['gti_total'].max():.2f} W/m²")
print("=" * 65)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# 1. Pull the computed physics matrix from your final Spark DataFrame
pdf_yield = df_final_power_forecast.orderBy("timestamp").toPandas()
pdf_yield["timestamp"] = pd.to_datetime(pdf_yield["timestamp"].dt.tz_localize(None))

# Create continuous index array to handle layout spacing
x_indices = np.arange(len(pdf_yield))

# 2. Initialize the visualization layout
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 7))

# ------------------------------------------------------------------
# PLOT THE ATTENUATED TILTED PLANE COMPONENTS
# ------------------------------------------------------------------
# Total Plane-of-Array Irradiance (GTI Global)
plt.plot(x_indices, pdf_yield["gti_total"], color="crimson", lw=2.5, 
         label="Total Tilted GTI (Global POA)")
plt.fill_between(x_indices, pdf_yield["gti_total"], color="crimson", alpha=0.08)

# Direct Beam component striking the panel face
plt.plot(x_indices, pdf_yield["gti_direct"], color="gold", lw=1.5, linestyle=":", 
         label="Direct Component on Panel")

# Scattered Sky component striking the panel face
plt.plot(x_indices, pdf_yield["gti_diffuse"], color="deepskyblue", lw=1.5, linestyle="--", 
         label="Diffuse Component on Panel")

# ------------------------------------------------------------------
# X-AXIS TIMELINE LABELLING OVERRIDES (7 Clean Spaced Milestones)
# ------------------------------------------------------------------
tick_indices = np.linspace(0, len(pdf_yield) - 1, 7, dtype=int)
tick_labels = pdf_yield["timestamp"].dt.strftime("%b %d").iloc[tick_indices].values

plt.xticks(tick_indices, tick_labels, fontsize=10)
plt.yticks(fontsize=10)

plt.title("7-Day Global Tilted Irradiance (GTI) Power Yield Time Series Profile (Perth Local Time)", 
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Irradiance ($W/m^2$)", fontsize=11, labelpad=10)
plt.xlabel("Forward Forecast Timeline Windows", fontsize=11, labelpad=10)
plt.ylim(-20, pdf_yield["gti_total"].max() * 1.08) # Dynamic padding for peak room
plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none", fontsize=10)

plt.tight_layout()
# Persist the figure to a Databricks Volume so scheduled jobs can publish it reliably.
# Create this volume once if needed: CREATE VOLUME IF NOT EXISTS main.default.`solar-figures`;
save_path = '/Volumes/main/default/solar-figures/7_Day_GTI_Power_Yield_Profile.png'
plt.savefig(save_path, bbox_inches='tight')
print(f"Forecast figure saved to: {save_path}")
plt.show()

In [0]:
import base64
from datetime import UTC, datetime

import requests

# Diagnostic write probe for Databricks Free Edition + GitHub API publishing.
# Token is stored in Databricks Secret Scope and is never printed.
GITHUB_TOKEN = dbutils.secrets.get(scope="github", key="GITHUB_TOKEN").strip()
GITHUB_REPO = "jun01ee/solar-yield-forecasting-pipeline"
GITHUB_BRANCH = "forecast-artifacts"
PROBE_PATH = "publish_probe.txt"
API_ROOT = "https://api.github.com"

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}


def raise_github_error(action, response):
    raise RuntimeError(
        f"{action} failed. status={response.status_code}, body={response.text}"
    )


def get_existing_file_sha(remote_path):
    url = f"{API_ROOT}/repos/{GITHUB_REPO}/contents/{remote_path}"
    response = requests.get(
        url,
        headers=GITHUB_HEADERS,
        params={"ref": GITHUB_BRANCH},
        timeout=30,
    )
    if response.status_code == 200:
        return response.json()["sha"]
    if response.status_code == 404:
        return None
    raise_github_error(f"Read GitHub file metadata for {remote_path}", response)


def put_github_file(remote_path, content_bytes, message):
    url = f"{API_ROOT}/repos/{GITHUB_REPO}/contents/{remote_path}"
    sha = get_existing_file_sha(remote_path)
    payload = {
        "message": message,
        "content": base64.b64encode(content_bytes).decode("utf-8"),
        "branch": GITHUB_BRANCH,
    }
    if sha:
        payload["sha"] = sha

    response = requests.put(url, headers=GITHUB_HEADERS, json=payload, timeout=60)
    if response.status_code not in (200, 201):
        raise_github_error(f"Write GitHub file {remote_path}", response)
    return response



repo_response = requests.get(
    f"{API_ROOT}/repos/{GITHUB_REPO}",
    headers=GITHUB_HEADERS,
    timeout=30,
)
if repo_response.status_code != 200:
    raise_github_error("Check GitHub repository access", repo_response)

permissions = repo_response.json().get("permissions", {})
print(f"GitHub repository access OK. Reported permissions: {permissions}")
if permissions and not permissions.get("push", False):
    raise RuntimeError(
        "GitHub token can read the repository but does not report push=True. "
        "Use a token with Contents: Read and write for this repository."
    )

branch_response = requests.get(
    f"{API_ROOT}/repos/{GITHUB_REPO}/branches/{GITHUB_BRANCH}",
    headers=GITHUB_HEADERS,
    timeout=30,
)
if branch_response.status_code != 200:
    raise_github_error(f"Check GitHub branch {GITHUB_BRANCH}", branch_response)
print(f"GitHub branch access OK: {GITHUB_BRANCH}")

probe_text = (
    "Databricks GitHub publish probe succeeded at "
    f"{datetime.now(UTC).isoformat()}\n"
)
put_github_file(
    PROBE_PATH,
    probe_text.encode("utf-8"),
    f"Update Databricks publish probe {datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S UTC')}",
)
print(f"GitHub write probe OK: {GITHUB_REPO}/{PROBE_PATH} on {GITHUB_BRANCH}")


In [0]:
import hashlib
import os
import shutil
import stat
import subprocess
from pathlib import Path

# Production publish: update the README forecast image on the artifact branch.
# This uses git CLI instead of REST upload for the PNG because large base64 JSON
# requests can fail with opaque 400 responses in restricted Databricks runtimes.
REMOTE_IMAGE_PATH = "7_Day_GTI_Power_Yield_Profile.png"
LOCAL_IMAGE_FILE = Path(save_path)
WORKDIR = Path("/tmp/solar_forecast_artifacts_repo")
ASKPASS_PATH = Path("/tmp/github_askpass_forecast.sh")

if not LOCAL_IMAGE_FILE.exists():
    raise FileNotFoundError(f"Forecast image was not created: {LOCAL_IMAGE_FILE}")

image_bytes = LOCAL_IMAGE_FILE.read_bytes()
print(f"Forecast image size: {len(image_bytes)} bytes")
print(f"Forecast image sha256: {hashlib.sha256(image_bytes).hexdigest()}")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

ASKPASS_PATH.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  *Username*) echo x-access-token ;;\n"
    "  *Password*) printf '%s\n' \"$GITHUB_TOKEN\" ;;\n"
    "  *) echo ;;\n"
    "esac\n"
)
ASKPASS_PATH.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(ASKPASS_PATH)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN

clone_url = f"https://github.com/{GITHUB_REPO}.git"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, clone_url, str(WORKDIR)],
    check=True,
    env=git_env,
)

remote_image = WORKDIR / REMOTE_IMAGE_PATH
shutil.copyfile(LOCAL_IMAGE_FILE, remote_image)

subprocess.run(["git", "config", "user.name", "Databricks Solar Forecast Bot"], cwd=WORKDIR, check=True)
subprocess.run(["git", "config", "user.email", "actions@users.noreply.github.com"], cwd=WORKDIR, check=True)
subprocess.run(["git", "add", REMOTE_IMAGE_PATH], cwd=WORKDIR, check=True)

commit_result = subprocess.run(
    ["git", "commit", "-m", f"Update daily solar forecast plot {datetime.now(UTC).strftime('%Y-%m-%d')}"],
    cwd=WORKDIR,
    text=True,
    capture_output=True,
)

if commit_result.returncode == 0:
    subprocess.run(["git", "push", "origin", GITHUB_BRANCH], cwd=WORKDIR, check=True, env=git_env)
    print(
        f"Published latest forecast image to {GITHUB_REPO}/{REMOTE_IMAGE_PATH} "
        f"on {GITHUB_BRANCH} via git CLI."
    )
elif "nothing to commit" in (commit_result.stdout + commit_result.stderr).lower():
    print("Forecast image is unchanged; no GitHub commit needed.")
else:
    raise RuntimeError(
        "Git commit for forecast image failed. "
        f"stdout={commit_result.stdout}, stderr={commit_result.stderr}"
    )

try:
    ASKPASS_PATH.unlink()
except FileNotFoundError:
    pass
